# Hisse Fiyat Tahmini (ARIMA)

Bu projede AAPL kapanış fiyatını tahmin edeceğim. Zaman serisi olduğu için hem ARIMA hem de lag'li modeller deneyeceğim.


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns


### Data


In [ ]:
df=pd.read_csv('data/AAPL.csv',parse_dates=['Date'])
df=df.sort_values('Date')
df.head()


### EDA


In [ ]:
df.info()
df.isnull().sum()


### Görselleştirme


In [ ]:
plt.plot(df['Date'],df['Close'])
plt.title('AAPL Close')
plt.show()


### Boş veri


In [ ]:
df['Close']=df['Close'].ffill()


### Feature Engineering


In [ ]:
s=df.copy()
s['lag1']=s['Close'].shift(1)
s['lag2']=s['Close'].shift(2)
s['lag3']=s['Close'].shift(3)
s=s.dropna()


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
x=s[['lag1','lag2','lag3']]
y=s['Close']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,shuffle=False)


### 3 Model


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error
from statsmodels.tsa.arima.model import ARIMA
lr=LinearRegression().fit(x_train,y_train)
rf=RandomForestRegressor(random_state=42).fit(x_train,y_train)
ar=ARIMA(y_train,order=(1,1,1)).fit()


In [ ]:
print('LR',round(r2_score(y_test,lr.predict(x_test)),3))


In [ ]:
print('RF',round(r2_score(y_test,rf.predict(x_test)),3))


In [ ]:
print('ARIMA',round(r2_score(y_test,ar.forecast(steps=len(y_test))),3))


### Feature Importance + Residual


In [ ]:
print(pd.Series(rf.feature_importances_,index=x.columns))
pred=lr.predict(x_test)
plt.scatter(pred,y_test-pred)
plt.axhline(0,color='r')
plt.show()


In [ ]:
import joblib
joblib.dump(lr,'../../models/timeseries_stock_arima.joblib')


### Sonuç

AAPL serisinde dünkü fiyat (lag1) çok belirleyici. Linear burada yeterli. Kısa seride ARIMA zorlandı. Hedefi kısmen tutturdum.
